# M45 lab solution: RAG or fine-tune

**Brief.** RAG or fine-tune: for one narrow task, compare the M10 RAG baseline with a small fine-tuned model on the same questions and write down which wins and why.

**The narrow task** is refund and billing questions about Larkspur Cloud. The test set is the eight golden questions of that task (`g01`, `g06`, `g08`, `g09`, `g14`, `g18`, `g25`, and the out-of-corpus `g26`) plus the off-topic probe from the M45 notebooks, *What do you think of Mars?*

**The two systems.**

- **RAG baseline (M10 style):** the handbook split with `RecursiveCharacterTextSplitter` (500 characters, 50 overlap), TF-IDF top 4 from `lab_kit`, and an LLM told to answer only from the excerpts.
- **Fine-tuned model:** `google/flan-t5-small` (small enough to train on a CPU) trained with the Hugging Face `Seq2SeqTrainer` on the 101 pairs of `../data/train.jsonl`, with no retrieval at answer time.

The base `flan-t5-small` is scored too, so the table shows what training changed.

**Scoring**, following the comparison table of lesson 45.6: an LLM judge marks each answer correct or incorrect against the golden `reference_answer`; token F1 against the same reference is a model-free check on the judge; a string check records whether each system declines the out-of-corpus and off-topic questions; a timer records seconds per answer. Then one handbook fact changes and both systems answer again, the RAG side after a re-index and the fine-tuned side without retraining.

Run with the `hf-foundations` group (`uv sync --group hf-foundations`). Needs `EXPERIENTIALLABS_API_KEY` for the RAG answers and the judge; the first run downloads `google/flan-t5-small` from the Hugging Face Hub.

In [ ]:
import re
import sys
import tempfile
import time
from collections import Counter
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.feature_extraction.text import TfidfVectorizer

load_dotenv()
sys.path.insert(0, str(Path("../../../../src").resolve()))
from helpers import get_experientiallabs_llm
from lab_kit import load_handbook, load_jsonl, normalise, tfidf_retriever

DATA = Path("../data")
pages = load_handbook(DATA / "larkspur" / "handbook")
golden = {g["id"]: g for g in load_jsonl(DATA / "larkspur" / "golden.jsonl")}
train_rows = load_jsonl(DATA / "train.jsonl")

TASK = ["g01", "g06", "g08", "g09", "g14", "g18", "g25", "g26"]
PROBE = {"id": "mars", "question": "What do you think of Mars?", "type": "off_topic",
         "evidence_spans": [], "reference_answer": "The handbook does not say."}
test = [golden[i] for i in TASK] + [PROBE]
print(f"{len(pages)} handbook pages, {len(train_rows)} training pairs, {len(test)} test questions")
for q in test:
    print(f"{q['id']:>5} {q['type']:<14} {q['question']}")

## 1. Check the training data against the test questions

No training question may be a golden question. The facts do overlap, on purpose: both systems learn from the same handbook. The cell prints the closest training question for each test question, so a near copy is visible before any score is read.

In [ ]:
train_q = {normalise(r["question"]) for r in train_rows}
copied = [q["id"] for q in test if normalise(q["question"]) in train_q]
assert not copied, f"test questions copied into train.jsonl: {copied}"

vec = TfidfVectorizer(stop_words="english").fit([r["question"] for r in train_rows] + [q["question"] for q in test])
sims = (vec.transform([q["question"] for q in test]) @ vec.transform([r["question"] for r in train_rows]).T).toarray()
for q, row in zip(test, sims):
    j = row.argmax()
    print(f"{q['id']:>5} {row[j]:.2f}  {q['question']}\n       nearest: {train_rows[j]['question']}")

## 2. The RAG baseline

The M10 baseline: recursive split, TF-IDF top 4, and a prompt that keeps the model to the excerpts. Each excerpt carries its handbook file name, so every answer can name its sources.

In [ ]:
DECLINE = "The handbook does not say."


def build_index(pages, k=4):
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunk_source = {c: p["source"] for p in pages for c in splitter.split_text(p["text"])}
    return tfidf_retriever(list(chunk_source), k), chunk_source


RAG_PROMPT = """You answer questions from Larkspur Cloud customers using only the handbook excerpts below.
If the excerpts do not answer the question, reply exactly: {decline}
Answer in one or two sentences.

Excerpts:
{context}

Question: {question}"""

llm = get_experientiallabs_llm()
search, chunk_source = build_index(pages)
print(f"{len(chunk_source)} chunks")


def rag_answer(question, search=search, chunk_source=chunk_source):
    start = time.perf_counter()
    chunks = search(question)
    context = "\n\n".join(f"[{chunk_source[c]}] {c}" for c in chunks)
    answer = llm.invoke(RAG_PROMPT.format(decline=DECLINE, context=context, question=question)).text.strip()
    return {"answer": answer, "sources": sorted({chunk_source[c] for c in chunks}), "seconds": time.perf_counter() - start}


answers = {"rag": {q["id"]: rag_answer(q["question"]) for q in test}}
for q in test:
    a = answers["rag"][q["id"]]
    print(f"{q['id']:>5} {a['sources']}\n       {a['answer']}")

## 3. Fine-tune `flan-t5-small`

Lessons 45.4 and 45.5 in one cell each: encode question and answer pairs, check what truncation would cut, then train with the `Trainer` API (`Seq2SeqTrainer`, the encoder-decoder version) and report the run in examples and epochs, not only steps. Ten percent of the pairs are held out for evaluation loss, as the M45 notebooks do.

Two checks from the M45 theory pages run before training: the tokenizer must append its end token (the `lamini_docs` rows had none, and the model never learned to stop), and `max_length = 64` must cut nothing.

In [ ]:
import torch
from datasets import Dataset
from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments, set_seed)

set_seed(0)
MODEL = "google/flan-t5-small"
PREFIX = "Answer the Larkspur Cloud customer question: "
MAX_LEN = 64
tokenizer = AutoTokenizer.from_pretrained(MODEL)

assert tokenizer("refund").input_ids[-1] == tokenizer.eos_token_id, "tokenizer does not append its end token"
longest_q = max(len(tokenizer(PREFIX + r["question"]).input_ids) for r in train_rows)
longest_a = max(len(tokenizer(text_target=r["answer"]).input_ids) for r in train_rows)
print(f"longest input {longest_q} tokens, longest answer {longest_a} tokens, max_length {MAX_LEN}")
assert max(longest_q, longest_a) <= MAX_LEN, "truncation would cut training rows"


def encode(row):
    enc = tokenizer(PREFIX + row["question"], truncation=True, max_length=MAX_LEN)
    enc["labels"] = tokenizer(text_target=row["answer"], truncation=True, max_length=MAX_LEN)["input_ids"]
    return enc


split = Dataset.from_list([{"question": r["question"], "answer": r["answer"]} for r in train_rows]).train_test_split(test_size=0.1, seed=0)
tokenized = split.map(encode, remove_columns=["question", "answer"])
print(tokenized)

In [ ]:
def t5_answer(model, question):
    start = time.perf_counter()
    inputs = tokenizer(PREFIX + question, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_LEN)
    return {"answer": tokenizer.decode(out[0], skip_special_tokens=True), "sources": [], "seconds": time.perf_counter() - start}


base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL).eval()
answers["base flan-t5-small"] = {q["id"]: t5_answer(base_model, q["question"]) for q in test}
for q in test:
    print(f"{q['id']:>5} {answers['base flan-t5-small'][q['id']]['answer']}")

In [ ]:
EPOCHS, BATCH, LR = 8, 8, 1e-3  # T5 models are usually fine-tuned at 1e-4 to 1e-3; eval loss per epoch below shows whether more epochs help

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)
args = Seq2SeqTrainingArguments(
    output_dir=tempfile.mkdtemp(prefix="m45_flan_t5_"),  # nothing is written inside the lab folder
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    learning_rate=LR,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=0,
)
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    processing_class=tokenizer,
)
start = time.perf_counter()
trainer.train()
train_seconds = time.perf_counter() - start

n = len(tokenized["train"])
print(f"{model.num_parameters():,} parameters")
print(f"{n} training examples x {EPOCHS} epochs = {n * EPOCHS} examples seen in {trainer.state.global_step} steps "
      f"of batch {BATCH}, {train_seconds:.0f} s on {model.device}")
print(pd.DataFrame([{"epoch": int(log["epoch"]), "eval_loss": round(log["eval_loss"], 3)}
                    for log in trainer.state.log_history if "eval_loss" in log]).to_string(index=False))

In [ ]:
model.eval()
answers["fine-tuned flan-t5-small"] = {q["id"]: t5_answer(model, q["question"]) for q in test}
for q in test:
    print(f"{q['id']:>5} {answers['fine-tuned flan-t5-small'][q['id']]['answer']}")

## 4. Score all three on the same questions

The judge sees the question, the golden reference and the answer, and returns one word. Token F1 is computed without a model, so a disagreement between the two is worth reading by hand. `declines` is a string check for the handbook's refusal sentence.

In [ ]:
JUDGE_PROMPT = """You grade answers from a customer-support assistant against a reference answer written from the company handbook.
The answer is CORRECT if it states the facts in the reference that answer the question and says nothing that contradicts it. Extra wording is fine.
If the reference says the handbook does not say, the answer is CORRECT only if it declines to answer.
Reply with one word: CORRECT or INCORRECT.

Question: {question}
Reference answer: {reference}
Answer to grade: {answer}"""
judge = get_experientiallabs_llm()


def judged_correct(q, answer):
    reply = judge.invoke(JUDGE_PROMPT.format(question=q["question"], reference=q["reference_answer"], answer=answer)).text
    return reply.strip().upper().startswith("CORRECT")


def tokens(text):
    return re.findall(r"[a-z0-9]+(?:[.%][0-9]+)?%?", normalise(text))


def token_f1(pred, ref):
    common = sum((Counter(tokens(pred)) & Counter(tokens(ref))).values())
    if not common:
        return 0.0
    p, r = common / len(tokens(pred)), common / len(tokens(ref))
    return 2 * p * r / (p + r)


def declines(answer):
    return "does not say" in normalise(answer)


rows = []
for system, by_id in answers.items():
    for q in test:
        a = by_id[q["id"]]
        rows.append({"system": system, "id": q["id"], "type": q["type"], "correct": judged_correct(q, a["answer"]),
                     "token_f1": round(token_f1(a["answer"], q["reference_answer"]), 2), "declines": declines(a["answer"]),
                     "cites": bool(a["sources"]), "seconds": round(a["seconds"], 2), "answer": a["answer"]})
per_question = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 120)
per_question

## 5. Change one fact and answer again

Lesson 45.6 ends with this test. The refund window in the handbook changes from 5 to 10 business days to 3 to 5. The RAG side is re-indexed from the changed text; the fine-tuned model is not retrained. Both answer `g01`, *How long does a refund take?*

In [ ]:
OLD, NEW = "take 5 to 10 business days", "take 3 to 5 business days"
changed = [dict(p, text=p["text"].replace(OLD, NEW)) for p in pages]
assert sum(OLD in p["text"] for p in pages) == 1 and sum(NEW in p["text"] for p in changed) == 1

search_changed, source_changed = build_index(changed)
question = golden["g01"]["question"]
after_change = {
    "rag (re-indexed)": rag_answer(question, search_changed, source_changed)["answer"],
    "fine-tuned flan-t5-small (not retrained)": t5_answer(model, question)["answer"],
}
fact_change = pd.DataFrame([{"system": s, "new window": "3 to 5" in normalise(a), "old window": "5 to 10" in normalise(a), "answer": a}
                            for s, a in after_change.items()])
fact_change

## Results

In [ ]:
task = per_question[per_question["id"].isin(TASK)]
summary = pd.DataFrame([
    {
        "system": system,
        f"judged correct (of {len(TASK)})": int(g["correct"].sum()),
        "mean token F1": round(g["token_f1"].mean(), 2),
        "declines g26": bool(g.loc[g["id"] == "g26", "declines"].iloc[0]),
        "declines Mars probe": bool(per_question[(per_question["system"] == system) & (per_question["id"] == "mars")]["declines"].iloc[0]),
        "cites sources": bool(g["cites"].all()),
        "mean seconds per answer": round(g["seconds"].mean(), 2),
    }
    for system, g in task.groupby("system", sort=False)
])
print(f"Models: RAG and judge {llm.model_name}; fine-tuned {MODEL}, {EPOCHS} epochs, {train_seconds:.0f} s of training")
summary

## Verdict

*Pending run.* The owner runs this notebook and writes the verdict here from the two tables above. M46's design document takes its RAG or fine-tune row from this verdict.

The comparison is set up to settle one question: for refund and billing answers, can a small model carry the handbook facts in its weights well enough to replace retrieval? Decide it with this rule, written before the run:

1. **Correctness.** The fine-tuned model has to match the RAG baseline's judged-correct count on the eight task questions. If it scores below, retrieval wins on the task itself.
2. **Refusals.** Both systems have to decline `g26` and the Mars probe. A system that invents a plan price or answers about Mars fails the task whatever its other scores.
3. **Currency.** The fact-change table decides where facts live. If the fine-tuned model still gives the old refund window, every policy change means retraining, while RAG needs a re-index.
4. **Cost.** Seconds per answer and the absence of an API call count for the fine-tuned model only once rules 1 to 3 are met.

Lesson 45.6 predicts which way it goes: RAG for facts that change, fine-tuning for behaviour such as style and refusals. This lab measures that prediction; it does not assume it.

## What this shows

*Pending run.* Write two or three sentences from the tables: where each system lost points, whether the judge and token F1 agree, and what the fact-change test says about putting refund policy in a model's weights.